## 환경설정


API KEY 를 설정합니다.


In [2]:
# API 키를 환경변수로 관리하기 위한 설정 파일
from dotenv import load_dotenv

# API 키 정보 로드
load_dotenv()

True

LangChain으로 구축한 애플리케이션은 여러 단계에 걸쳐 LLM 호출을 여러 번 사용하게 됩니다. 이러한 애플리케이션이 점점 더 복잡해짐에 따라, 체인이나 에이전트 내부에서 정확히 무슨 일이 일어나고 있는지 조사할 수 있는 능력이 매우 중요해집니다. 이를 위한 최선의 방법은 [LangSmith](https://smith.langchain.com)를 사용하는 것입니다.

LangSmith가 필수는 아니지만, 유용합니다. LangSmith를 사용하고 싶다면, 위의 링크에서 가입한 후, 로깅 추적을 시작하기 위해 환경 변수를 설정해야 합니다.


## 모듈별로 자세히 살펴보기


In [3]:
import bs4
from langchain import hub
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import WebBaseLoader
from langchain_community.vectorstores import Chroma, FAISS
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from langchain_openai import ChatOpenAI, OpenAIEmbeddings

USER_AGENT environment variable not set, consider setting it to identify your requests.


아래는 [](https://teddylee777.github.io/langchain/rag-naver-news-qa/) 에서 다뤘던 기본적인 RAG 모델을 사용하는 예제입니다.

여기서 각 단계별로 다양한 옵션을 설정하거나 새로운 기법을 적용할 수 있습니다.


In [5]:
# 단계 1: 문서 로드(Load Documents)
# 뉴스기사 내용을 로드하고, 청크로 나누고, 인덱싱합니다.
url = "https://n.news.naver.com/article/437/0000378416"
loader = WebBaseLoader(
    web_paths=(url,),
    bs_kwargs=dict(
        parse_only=bs4.SoupStrainer(
            "div",
            attrs={"class": ["newsct_article _article_body", "media_end_head_title"]},
        )
    ),
)
docs = loader.load()


# 단계 2: 문서 분할(Split Documents)
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=50)

splits = text_splitter.split_documents(docs)

# 단계 3: 임베딩 & 벡터스토어 생성(Create Vectorstore)
# 벡터스토어를 생성합니다.
vectorstore = FAISS.from_documents(documents=splits, embedding=OpenAIEmbeddings())

# 단계 4: 검색(Search)
# 뉴스에 포함되어 있는 정보를 검색하고 생성합니다.
retriever = vectorstore.as_retriever()

# 단계 5: 프롬프트 생성(Create Prompt)
# 프롬프트를 생성합니다.
prompt = hub.pull("rlm/rag-prompt")

# 단계 6: 언어모델 생성(Create LLM)
# 모델(LLM) 을 생성합니다.
llm = ChatOpenAI(model_name="gpt-3.5-turbo", temperature=0)


def format_docs(docs):
    # 검색한 문서 결과를 하나의 문단으로 합쳐줍니다.
    return "\n\n".join(doc.page_content for doc in docs)


# 단계 7: 체인 생성(Create Chain)
rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

# 단계 8: 체인 실행(Run Chain)
# 문서에 대한 질의를 입력하고, 답변을 출력합니다.
question = "부영그룹의 출산 장려 정책에 대해 설명해주세요"
response = rag_chain.invoke(question)

# 결과 출력
print(f"URL: {url}")
print(f"문서의 수: {len(docs)}")
print("===" * 20)
print(f"[HUMAN]\n{question}\n")
print(f"[AI]\n{response}")

ImportError: Could not import faiss python package. Please install it with `pip install faiss-gpu` (for CUDA supported GPU) or `pip install faiss-cpu` (depending on Python version).

## 단계 1: 문서 로드(Load Documents)

- [공식문서 링크 - Document loaders](https://python.langchain.com/docs/modules/data_connection/document_loaders/)


In [5]:
import json
import pandas as pd
from datetime import datetime

with open ("instagram_diary/content/posts_1.json", "r") as f:
    data = json.load(f)

# DataFrame 생성
df = pd.DataFrame()
for i in range(len(data)):
    df = pd.concat([df,pd.DataFrame(data[i])])

# 유니코드 이스케이프 시퀀스를 디코딩
df = df.map(lambda x: x.encode('latin-1').decode('utf-8') if isinstance(x, str) else x)
df = df.reset_index(drop=True)

#  전처리 


# df = df.copy()
df.loc[df.title.isna(),'creation_timestamp']= [x.get('creation_timestamp') for x in df.loc[df.title.isna()]['media']]
df.loc[df.title.isna(),'title']= [x.get('title').encode('latin-1').decode('utf-8') for x in df.loc[df.title.isna()]['media']]
df['dt'] = [datetime.fromtimestamp(x) for x in df['creation_timestamp']]

# 결과 출력
print(df.shape)
df.head()


(4455, 4)


,media,title,creation_timestamp,dt
0,{'uri': 'media/posts/202407/451325293_83729456...,"인도네시아로 떠나버린 엄마와 영종도로 떠나버린 언니,,,아침부터 아주 시끌벅적 난리...",1.721013e+09,2024-07-15 12:17:39
1,{'uri': 'media/posts/202407/450766642_98856626...,"인도네시아로 떠나버린 엄마와 영종도로 떠나버린 언니,,,아침부터 아주 시끌벅적 난리...",1.721013e+09,2024-07-15 12:17:39
2,{'uri': 'media/posts/202407/451295091_12016493...,"인도네시아로 떠나버린 엄마와 영종도로 떠나버린 언니,,,아침부터 아주 시끌벅적 난리...",1.721013e+09,2024-07-15 12:17:39
3,{'uri': 'media/posts/202407/451268789_47500573...,"인도네시아로 떠나버린 엄마와 영종도로 떠나버린 언니,,,아침부터 아주 시끌벅적 난리...",1.721013e+09,2024-07-15 12:17:39
4,{'uri': 'media/posts/202407/451422413_45867830...,"인도네시아로 떠나버린 엄마와 영종도로 떠나버린 언니,,,아침부터 아주 시끌벅적 난리...",1.721013e+09,2024-07-15 12:17:39


In [6]:
df_nodup = df[["title", 'dt']].drop_duplicates(keep='last')
print(df_nodup.shape)
df_nodup.head()

(1449, 2)


,title,dt
4,"인도네시아로 떠나버린 엄마와 영종도로 떠나버린 언니,,,아침부터 아주 시끌벅적 난리...",2024-07-15 12:17:39
5,아니 오늘 사진 이거 밖에 없어? 근데 오늘 정말 웃기는 하루ㅋㅋㅋ 아침에 만나서 ...,2024-07-13 23:56:29
8,퇴근하구 오빠랑 양재에서 술막엇다. 출근해서 왠지모르게 자꾸 맛잇는가 먹고싶은 날이...,2024-07-13 01:56:38
13,오늘 요약 재밋엇음 은근 사람만나는것도 재밋네 혼자 노는것도 물론 재밋긴한데.ㅋㅋㅋ...,2024-07-11 23:39:17
18,ㅎㅏ 별개로 누님과의 만남은 즐거웠움 김유환 호감도-100 김은수 호감도+10000...,2024-07-11 00:11:55


In [11]:
from langchain_community.document_loaders import DataFrameLoader
# DataFrame에서 데이터를 로드하고, 'Team' 열을 페이지 내용으로 사용합니다.
loader = DataFrameLoader(df_nodup, page_content_column="title")

docs = loader.load()  # 데이터를 로드합니다.
print(f"문서의 수: {len(docs)}\n")

# 10번째 페이지의 내용 출력
print(f"\n[페이지내용]\n{docs[1400].page_content[:500]}")
print(f"\n[metadata]\n{docs[1400].metadata}\n")


문서의 수: 1449


[페이지내용]
마음에 들었다 슬랙스!

[metadata]
{'dt': Timestamp('2016-05-06 15:42:46')}



---


## 단계 2: 문서 분할(Split Documents)


### RecursiveTextSplitter

이 텍스트 분할기는 일반 텍스트에 권장되는 텍스트 분할기입니다.

1. 텍스트가 어떻게 분할 규칙: list of `separators`
2. 청크 크기가 어떻게 측정되는가: `len` of characters


In [15]:
# langchain 패키지에서 RecursiveCharacterTextSplitter 클래스를 가져옵니다.
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.text_splitter import CharacterTextSplitter

`RecursiveCharacterTextSplitter` 클래스는 텍스트를 재귀적으로 분할하는 기능을 제공합니다. 이 클래스는 `chunk_size`로 분할할 청크의 크기, `chunk_overlap`으로 인접 청크 간의 겹침 크기, `length_function`으로 청크의 길이를 계산하는 함수, 그리고 `is_separator_regex`로 구분자가 정규 표현식인지 여부를 지정하는 매개변수를 받습니다. 예시에서는 청크 크기를 100, 겹침 크기를 20으로 설정하고, 길이 계산 함수로 `len`을 사용하며, 구분자가 정규 표현식이 아님을 나타내기 위해 `is_separator_regex`를 `False`로 설정합니다.


In [13]:
recursive_text_splitter = RecursiveCharacterTextSplitter(
    # 정말 작은 청크 크기를 설정합니다.
    chunk_size=100,
    chunk_overlap=10,
    length_function=len,
    is_separator_regex=False,
)

In [16]:
character_text_splitter = CharacterTextSplitter(
    chunk_size=100, chunk_overlap=10, separator=" "
)
for sent in character_text_splitter.split_documents(docs):
    print(sent)
print("===" * 20)
recursive_text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=100, chunk_overlap=10
)
for sent in recursive_text_splitter.split_documents(docs):
    print(sent)

page_content='인도네시아로 떠나버린 엄마와 영종도로 떠나버린 언니,,,아침부터 아주 시끌벅적 난리엿다 겨우 엄마랑 인사하고 쉬즈곤. 잘다녀오세용~그래서 갑자기 결정된 오빠 우리집' metadata={'dt': Timestamp('2024-07-15 12:17:39')}
page_content='결정된 오빠 우리집 놀러오기~~~오빠가 아침부터 와줘서 점심도 까르보나라 해줫다. 웃겼던건 오빠 치즈 열심히 다지고 잇엇는데 내가 강판으로 갈아버리기ㅠ미안해 난오빠가 강판' metadata={'dt': Timestamp('2024-07-15 12:17:39')}
page_content='난오빠가 강판 싫어하는줄 알았어,,,오늘도 역시 레전드 맛잇는 까르보나라^____^* 행복해랏~~~ 김유환 내가 까르보나라 잘하는 거 같으니까 은근히 서운해하는게' metadata={'dt': Timestamp('2024-07-15 12:17:39')}
page_content='은근히 서운해하는게 웃기다ㅋㅋㅋㅋㅋㅋㅋㅋㅋ하지만 오빠가 해주는게 젤로 맛잇다고요~~~^___^* 나는 옆에서 오빠 요리 보조햇다. 그냥 옆에서 이거저거 치워주기 그러고 아 들어오기' metadata={'dt': Timestamp('2024-07-15 12:17:39')}
page_content='그러고 아 들어오기 전에 마트에서 오겹살 사서 에어프라이어에 구웟다ㅋㅋㅋㅋㅋ어떻게 이런 멋진 생각을! 아주 조은 생각이야 근데 좀 많앗다. 아 밥먹으면서 푸른산호초 부르는 하니에' metadata={'dt': Timestamp('2024-07-15 12:17:39')}
page_content='부르는 하니에 빠져서 뉴진스를 진짜 무한반복해서 영상봄 첨엔 너무 예쁘고 귀엽고 즐겁고 재밌었는데 김유환이 내가 뉴진스 예쁘다고 하는 말에 동조하자마자 재미없어짐. 보다가 점점' metadata={'dt': Timestamp('2024-07-15 12:17:39')}
page_content='보다가 점점 지겨워짐 이러다가 

- 지정한 separators 리스트를 순차적으로 시도하며 주어진 문서를 분할합니다.
- 청크가 충분히 작아질 때까지 순서대로 분할을 시도합니다. 기본 목록은 ["\n\n", "\n", " ", ""]입니다.
- 이는 일반적으로 의미적으로 가장 연관성이 강한 텍스트 조각인 것처럼 보이는 모든 단락(그리고 문장, 단어)을 가능한 한 길게 유지하려는 효과가 있습니다.


In [17]:
# recursive_text_splitter 에 기본 지정된 separators 를 확인합니다.
recursive_text_splitter._separators

['\n\n', '\n', ' ', '']

In [41]:
loader = DataFrameLoader(df_nodup, page_content_column="title")

# splitter 를 정의합니다.
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000,
                                                chunk_overlap=50,
                                                length_function=len,
                                                is_separator_regex=False,)

# 문서를 로드시 바로 분할까지 수행합니다.
split_docs = loader.load_and_split(text_splitter=text_splitter)
print(f"문서의 수: {len(docs)}")
docs[14].page_content[:500]

문서의 수: 1449


'엽떡데이-저스틴 왜 잔을 안들고오새용?🙂 엽떳 과식하고 저녁은 굶어버려 퇴근해서 낼 도시락준비하고 누워서 쉬엇다. 목이 넘 간지러'

### Semantic Similarity



의미적 유사성을 기준으로 텍스트를 분할합니다.

출처: [Greg Kamradt’s Notebook](https://github.com/FullStackRetrieval-com/RetrievalTutorials/blob/main/5_Levels_Of_Text_Splitting.ipynb)

높은 수준(high level)에서 문장으로 분할한 다음 3개 문장으로 그룹화한 다음 임베딩 공간에서 유사한 문장을 병합하는 방식입니다.


In [ ]:
# 최신 버전으로 업데이트합니다.
# !pip install -U langchain langchain_experimental -q

In [2]:
from langchain_experimental.text_splitter import SemanticChunker
from langchain_openai.embeddings import OpenAIEmbeddings

# SemanticChunker 를 생성합니다.
semantic_text_splitter = SemanticChunker(OpenAIEmbeddings(), add_start_index=True)

In [1]:
# chain of density 논문의 일부 내용을 불러옵니다
for sent in semantic_text_splitter.split_text(docs):
    print(sent)
    print("===" * 20)

NameError: name 'semantic_text_splitter' is not defined

In [ ]:
# chain of density 논문의 일부 내용을 불러옵니다
with open("data/chain-of-density.txt", "r") as f:
    text = f.read()

for sent in semantic_text_splitter.split_text(text):
    print(sent)
    print("===" * 20)

In [ ]:
from langchain_community.vectorstores import FAISS
from langchain_openai.embeddings import OpenAIEmbeddings

# 단계 3: 임베딩 & 벡터스토어 생성(Create Vectorstore)
# 벡터스토어를 생성합니다.
vectorstore = FAISS.from_documents(documents=splits, embedding=OpenAIEmbeddings())

## 3 단계: 임베딩

참고: https://python.langchain.com/docs/integrations/text_embedding


### 유료 과금 임베딩(OpenAI)


In [ ]:
from langchain_community.vectorstores import FAISS
from langchain_openai.embeddings import OpenAIEmbeddings

# 단계 3: 임베딩 & 벡터스토어 생성(Create Vectorstore)
# 벡터스토어를 생성합니다.
vectorstore = FAISS.from_documents(documents=splits, embedding=OpenAIEmbeddings())

다음은 `OpenAI` 의 지원되는 Embedding 모델들의 목록입니다.

- 기본 값은 `text-embeding-ada-002` 입니다.


| MODEL                  | ROUGH PAGES PER DOLLAR | EXAMPLE PERFORMANCE ON MTEB EVAL |
| ---------------------- | ---------------------- | -------------------------------- |
| text-embedding-3-small | 62,500                 | 62.3%                            |
| text-embedding-3-large | 9,615                  | 64.6%                            |
| text-embedding-ada-002 | 12,500                 | 61.0%                            |


In [ ]:
vectorstore = FAISS.from_documents(
    documents=splits, embedding=OpenAIEmbeddings(model="text-embedding-3-small")
)

### 무료 Open Source 기반 임베딩


In [26]:
from langchain_community.embeddings import HuggingFaceBgeEmbeddings

# 단계 3: 임베딩 & 벡터스토어 생성(Create Vectorstore)
# 벡터스토어를 생성합니다.
vectorstore = FAISS.from_documents(
    documents=split_docs, embedding=HuggingFaceBgeEmbeddings()
)

model.safetensors:  59%|#####9    | 797M/1.34G [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/366 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

1_Pooling/config.json:   0%|          | 0.00/191 [00:00<?, ?B/s]

In [ ]:
# !pip install fastembed -q

In [ ]:
from langchain_community.embeddings.fastembed import FastEmbedEmbeddings

vectorstore = FAISS.from_documents(documents=splits, embedding=FastEmbedEmbeddings())

## 4단계: 벡터스토어 생성(Create Vectorstore)


In [ ]:
from langchain_community.vectorstores import FAISS

# FAISS DB 적용
vectorstore = FAISS.from_documents(documents=splits, embedding=OpenAIEmbeddings())

In [ ]:
from langchain_community.vectorstores import Chroma

# Chroma DB 적용
vectorstore = Chroma.from_documents(documents=splits, embedding=OpenAIEmbeddings())

## 5단계: Retriever 생성

리트리버는 구조화되지 않은 쿼리가 주어지면 문서를 반환하는 인터페이스입니다.

리트리버는 문서를 저장할 필요 없이 문서를 반환(또는 검색)하기만 합니다.

- [공식 도큐먼트](https://python.langchain.com/docs/modules/data_connection/retrievers/)

생성된 VectorStore 에 `as_retriver()` 로 가져와서 **Retriever** 를 생성합니다.


### 유사도 기반 검색

- 기본값은 코사인 유사도인 `similarity` 가 적용되어 있습니다.


In [42]:
query = "2022년도엔 대체로 무슨일이 있었어?"

retriever = vectorstore.as_retriever(search_type="similarity")
search_result = retriever.get_relevant_documents(query)
print(search_result)

[Document(metadata={'dt': Timestamp('2022-12-31 00:34:25')}, page_content='이얏호 2022마지막 출근-잠실에서 술먹기'), Document(metadata={'dt': Timestamp('2023-12-31 18:35:00')}, page_content='아침에 딩굴대다가 전부치는 하루 갈비구워먹을거 약간 기대햇는데 고모가 해오신대서 우린 안해도됐다 근데 우리가 하는 갈비가 더 맛잇는데..ㅠ.ㅠ그래도 대충 네시쯤 끝낫으니까 금방끝난듯 낼 가면 새뱃돈 주시나? 내가 사촌들 중에 젤 짧게 받앗어 너무 손해야 문석오빠는 35까지 받앗는데 나만 못받아 나는 줘야지!! 이불 바꾸고 쉬니까 행복하다 배부르고 등따신 2023 마지막날 아 정리하는@릴스@만들어야겟다'), Document(metadata={'dt': Timestamp('2023-12-27 04:10:19')}, page_content='좋아 도대체 뮤슨일일까? 나에게 이런 방탕한 삶이 허락되는 날이 오다니 아무도 허락안햇지만 그냥 나혼자 폭주하기 내가 허락하면 됐지 뭐,,진짜 근데 너무너무 방탕하고 오빠가 너무 잘해줘서 거의 목욕시중을 들어주는 수준인데 몸 둘바를 모르겠기도 하고 근데 또 너무 좋고 야하고 근데 또 못 움직이겟는 것도 맞아ㅋㅋㅋ체력이..자꾸 바닥나는데 오빠가 마사지해주고 목욕시켜줘서 강제로 끌어올리고 난 아 다 회복됐나보다^^하고 착각하고 다시 뽈뽈거리다가 체력 바닥나고 오빠가 다시 강제로 끌어올리고ㅋㅋㅋㅋㅋㅋㅋㅋㅋㅋ이 순환이 너무 웃기다. 행복하고 즐거워ㅋㅋㅋ운동 열심히해야겠다..!!'), Document(metadata={'dt': Timestamp('2021-06-02 21:45:02')}, page_content='대학원 도서관의 하루. 거기서 뭘했는진 모르겠지만?? 그냥 공간이 예뻐서 기분이 좋았다. 낼 면접인데 왕창 스트레스 받다가 또 그냥 아 엔씨 구경하고 와야지..하고 그냥 맘놓기. 난 쓰레기같은 삶살지만 조금이라도 덜 

`similarity_score_threshold` 는 유사도 기반 검색에서 `score_threshold` 이상인 결과만 반환합니다.


In [ ]:
query = "회사의 저출생 정책이 뭐야?"

retriever = vectorstore.as_retriever(
    search_type="similarity_score_threshold", search_kwargs={"score_threshold": 0.8}
)
search_result = retriever.get_relevant_documents(query)
print(search_result)

`maximum marginal search result` 를 사용하여 검색합니다.


In [ ]:
query = "회사의 저출생 정책이 뭐야?"

retriever = vectorstore.as_retriever(search_type="mmr", search_kwargs={"k": 2})
search_result = retriever.get_relevant_documents(query)
print(search_result)

### 다양한 쿼리 생성


In [28]:
from langchain.retrievers.multi_query import MultiQueryRetriever
from langchain_openai import ChatOpenAI

query = "22년도엔 무슨일이 있었어?"

llm = ChatOpenAI(temperature=0, model="gpt-3.5-turbo")

retriever_from_llm = MultiQueryRetriever.from_llm(
    retriever=vectorstore.as_retriever(), llm=llm
)

In [ ]:
# Set logging for the queries
import logging

logging.basicConfig()
logging.getLogger("langchain.retrievers.multi_query").setLevel(logging.INFO)

In [ ]:
unique_docs = retriever_from_llm.get_relevant_documents(query=question)
len(unique_docs)

### Ensemble Retriever


In [44]:
from langchain.retrievers import BM25Retriever, EnsembleRetriever
from langchain_community.vectorstores import FAISS
from langchain_openai import OpenAIEmbeddings

In [46]:
doc_list = [
    "난 오늘 많이 먹어서 배가 정말 부르다",
    "떠나는 저 배가 오늘 마지막 배인가요?",
    "내가 제일 좋아하는 과일들은 배, 사과, 키워, 수박 입니다.",
]

# initialize the bm25 retriever and faiss retriever
bm25_retriever = BM25Retriever.from_texts(doc_list)
bm25_retriever.k = 2

faiss_vectorstore = FAISS.from_texts(doc_list, OpenAIEmbeddings())
faiss_retriever = faiss_vectorstore.as_retriever(search_kwargs={"k": 2})

# initialize the ensemble retriever
ensemble_retriever = EnsembleRetriever(
    retrievers=[bm25_retriever, faiss_retriever], weights=[0.5, 0.5]
)

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


In [47]:
def pretty_print(docs):
    for i, doc in enumerate(docs):
        print(f"[{i+1}] {doc.page_content}")

In [48]:
sample_query = "나 요즘 배에 정말 살이 많이 쪘어..."
print(f"[Query]\n{sample_query}\n")
relevant_docs = bm25_retriever.get_relevant_documents(sample_query)
print("[BM25 Retriever]")
pretty_print(relevant_docs)
print("===" * 20)
relevant_docs = faiss_retriever.get_relevant_documents(sample_query)
print("[FAISS Retriever]")
pretty_print(relevant_docs)
print("===" * 20)
relevant_docs = ensemble_retriever.get_relevant_documents(sample_query)
print("[Ensemble Retriever]")
pretty_print(relevant_docs)

[Query]
나 요즘 배에 정말 살이 많이 쪘어...

[BM25 Retriever]
[1] 난 오늘 많이 먹어서 배가 정말 부르다
[2] 내가 제일 좋아하는 과일들은 배, 사과, 키워, 수박 입니다.
[FAISS Retriever]
[1] 난 오늘 많이 먹어서 배가 정말 부르다
[2] 떠나는 저 배가 오늘 마지막 배인가요?
[Ensemble Retriever]
[1] 난 오늘 많이 먹어서 배가 정말 부르다
[2] 내가 제일 좋아하는 과일들은 배, 사과, 키워, 수박 입니다.
[3] 떠나는 저 배가 오늘 마지막 배인가요?


In [49]:
sample_query = "바다 위에 떠다니는 배들이 많다"
print(f"[Query]\n{sample_query}\n")
relevant_docs = bm25_retriever.get_relevant_documents(sample_query)
print("[BM25 Retriever]")
pretty_print(relevant_docs)
print("===" * 20)
relevant_docs = faiss_retriever.get_relevant_documents(sample_query)
print("[FAISS Retriever]")
pretty_print(relevant_docs)
print("===" * 20)
relevant_docs = ensemble_retriever.get_relevant_documents(sample_query)
print("[Ensemble Retriever]")
pretty_print(relevant_docs)

[Query]
바다 위에 떠다니는 배들이 많다

[BM25 Retriever]
[1] 내가 제일 좋아하는 과일들은 배, 사과, 키워, 수박 입니다.
[2] 떠나는 저 배가 오늘 마지막 배인가요?
[FAISS Retriever]
[1] 난 오늘 많이 먹어서 배가 정말 부르다
[2] 떠나는 저 배가 오늘 마지막 배인가요?
[Ensemble Retriever]
[1] 떠나는 저 배가 오늘 마지막 배인가요?
[2] 내가 제일 좋아하는 과일들은 배, 사과, 키워, 수박 입니다.
[3] 난 오늘 많이 먹어서 배가 정말 부르다


In [50]:
sample_query = "ships"
print(f"[Query]\n{sample_query}\n")
relevant_docs = bm25_retriever.get_relevant_documents(sample_query)
print("[BM25 Retriever]")
pretty_print(relevant_docs)
print("===" * 20)
relevant_docs = faiss_retriever.get_relevant_documents(sample_query)
print("[FAISS Retriever]")
pretty_print(relevant_docs)
print("===" * 20)
relevant_docs = ensemble_retriever.get_relevant_documents(sample_query)
print("[Ensemble Retriever]")
pretty_print(relevant_docs)

[Query]
ships

[BM25 Retriever]
[1] 내가 제일 좋아하는 과일들은 배, 사과, 키워, 수박 입니다.
[2] 떠나는 저 배가 오늘 마지막 배인가요?
[FAISS Retriever]
[1] 떠나는 저 배가 오늘 마지막 배인가요?
[2] 내가 제일 좋아하는 과일들은 배, 사과, 키워, 수박 입니다.
[Ensemble Retriever]
[1] 내가 제일 좋아하는 과일들은 배, 사과, 키워, 수박 입니다.
[2] 떠나는 저 배가 오늘 마지막 배인가요?


In [ ]:
sample_query = "pear"
print(f"[Query]\n{sample_query}\n")
relevant_docs = bm25_retriever.get_relevant_documents(sample_query)
print("[BM25 Retriever]")
pretty_print(relevant_docs)
print("===" * 20)
relevant_docs = faiss_retriever.get_relevant_documents(sample_query)
print("[FAISS Retriever]")
pretty_print(relevant_docs)
print("===" * 20)
relevant_docs = ensemble_retriever.get_relevant_documents(sample_query)
print("[Ensemble Retriever]")
pretty_print(relevant_docs)

## 6단계: 프롬프트 생성(Create Prompt)


프롬프트 엔지니어링은 주어진 데이터(`context`)를 토대로 우리가 원하는 결과를 도출할 때 중요한 역할을 합니다.

[TIP1]

1. 만약, `retriever` 에서 도출한 결과에서 중요한 정보가 누락된다면 `retriever` 의 로직을 수정해야 합니다.
2. 만약, `retriever` 에서 도출한 결과가 많은 정보를 포함하고 있지만, `llm` 이 그 중에서 중요한 정보를 찾지 못한거나 원하는 형태로 출력하지 않는다면 프롬프트를 수정해야 합니다.

[TIP2]

1. LangSmith 의 **hub** 에는 검증된 프롬프트가 많이 업로드 되어 있습니다.
2. 검증된 프롬프트를 활용하거나 약간 수정한다면 비용과 시간을 절약할 수 있습니다.

- https://smith.langchain.com/hub/search?q=rag


In [ ]:
from langchain import hub

In [51]:
prompt = hub.pull("rlm/rag-prompt")
prompt

ChatPromptTemplate(input_variables=['context', 'question'], metadata={'lc_hub_owner': 'rlm', 'lc_hub_repo': 'rag-prompt', 'lc_hub_commit_hash': '50442af133e61576e74536c6556cefe1fac147cad032f4377b60c436e6cdcb6e'}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context', 'question'], template="You are an assistant for question-answering tasks. Use the following pieces of retrieved context to answer the question. If you don't know the answer, just say that you don't know. Use three sentences maximum and keep the answer concise.\nQuestion: {question} \nContext: {context} \nAnswer:"))])

## 7단계: 언어모델 생성(Create LLM)


OpenAI 모델 중 하나를 선택합니다.

- `gpt-3.5-turbo` : OpenAI의 GPT-3.5-turbo 모델
- `gpt-4-turbo-preview` : OpenAI의 GPT-4-turbo-preview 모델

자세한 비용 체계는 [OpenAI API 모델 리스트 / 요금표](https://teddylee777.github.io/openai/openai-models/)에서 확인할 수 있습니다.


In [ ]:
from langchain_openai import ChatOpenAI

model = ChatOpenAI(temperature=0, model="gpt-3.5-turbo")

다음의 방식으로 토큰 사용량을 확인할 수 있습니다.


In [ ]:
from langchain.callbacks import get_openai_callback

with get_openai_callback() as cb:
    result = model.invoke("대한민국의 수도는 어디인가요?")
print(cb)

HuggingFace 에 업로드 되어 있는 오픈소스 모델을 손쉽게 다운로드 받아 사용할 수 있습니다.

아래의 리더보드에서 날마다 성능을 개선하는 오픈소스 리더보드를 확인할 수 있습니다.

- [HuggingFace LLM Leaderboard](https://huggingface.co/spaces/HuggingFaceH4/open_llm_leaderboard)


In [ ]:
# HuggingFaceHub 객체 생성
from langchain.llms import HuggingFaceHub

repo_id = "google/flan-t5-xxl"

t5_model = HuggingFaceHub(
    repo_id=repo_id, model_kwargs={"temperature": 0.1, "max_length": 512}
)

In [ ]:
t5_model.invoke("Where is the capital of South Korea?")

## RAG 템플릿 실험


In [ ]:
# 단계 1: 문서 로드(Load Documents)
# 문서를 로드하고, 청크로 나누고, 인덱싱합니다.
from langchain.document_loaders import PyPDFLoader

# PDF 파일 로드. 파일의 경로 입력
file_path = "data/SPRI_AI_Brief_2023년12월호_F.pdf"
loader = PyPDFLoader(file_path=file_path)

# 단계 2: 문서 분할(Split Documents)
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=50)

split_docs = loader.load_and_split(text_splitter=text_splitter)

# 단계 3, 4: 임베딩 & 벡터스토어 생성(Create Vectorstore)
# 벡터스토어를 생성합니다.
# vectorstore = FAISS.from_documents(documents=splits, embedding=OpenAIEmbeddings())
vectorstore = FAISS.load_local('../db/faiss', HuggingFaceBgeEmbeddings(), allow_dangerous_deserialization=True)

# 단계 5: 리트리버 생성(Create Retriever)
# 사용자의 질문(query) 에 부합하는 문서를 검색합니다.

# 유사도 높은 K 개의 문서를 검색합니다.
k = 3

# (Sparse) bm25 retriever and (Dense) faiss retriever 를 초기화 합니다.
bm25_retriever = BM25Retriever.from_documents(split_docs)
bm25_retriever.k = k

faiss_vectorstore = FAISS.from_documents(split_docs, OpenAIEmbeddings())
faiss_retriever = faiss_vectorstore.as_retriever(search_kwargs={"k": k})

# initialize the ensemble retriever
ensemble_retriever = EnsembleRetriever(
    retrievers=[bm25_retriever, faiss_retriever], weights=[0.5, 0.5]
)

# 단계 6: 프롬프트 생성(Create Prompt)
# 프롬프트를 생성합니다.
prompt = hub.pull("rlm/rag-prompt")

# 단계 7: 언어모델 생성(Create LLM)
# 모델(LLM) 을 생성합니다.
llm = ChatOpenAI(model_name="gpt-3.5-turbo", temperature=0)


def format_docs(docs):
    # 검색한 문서 결과를 하나의 문단으로 합쳐줍니다.
    return "\n\n".join(doc.page_content for doc in docs)


# 단계 8: 체인 생성(Create Chain)
rag_chain = (
    {"context": ensemble_retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

# 결과 출력
print(f"PDF Path: {file_path}")
print(f"문서의 수: {len(docs)}")
print("===" * 20)
print(f"[HUMAN]\n{question}\n")
print(f"[AI]\n{response}")

> 문서: data/SPRI_AI_Brief_2023년12월호\_F.pdf (페이지 10)

- LangSmith: https://smith.langchain.com/public/4449e744-f0a0-42d2-a3df-855bd7f41652/r


In [53]:
# 단계 8: 체인 실행(Run Chain)
# 문서에 대한 질의를 입력하고, 답변을 출력합니다.
question = "삼성 가우스에 대해 설명해주세요"
response = rag_chain.invoke(question)
print(response)

NameError: name 'rag_chain' is not defined

> 문서: data/SPRI_AI_Brief_2023년12월호\_F.pdf (페이지 12)

- LangSmith: https://smith.langchain.com/public/2b2913c9-6b9c-4a19-bb16-dc2256e2fdbf/r


In [ ]:
# 단계 8: 체인 실행(Run Chain)
# 문서에 대한 질의를 입력하고, 답변을 출력합니다.
question = "미래의 AI 소프트웨어 매출 전망은 어떻게 되나요?"
response = rag_chain.invoke(question)
print(response)

> 문서: data/SPRI_AI_Brief_2023년12월호\_F.pdf (페이지 14)

- LangSmith: https://smith.langchain.com/public/17ef6df2-b012-4f8e-b0a8-62894d82c097/r


In [ ]:
# 단계 8: 체인 실행(Run Chain)
# 문서에 대한 질의를 입력하고, 답변을 출력합니다.
question = "YouTube 가 2024년에 의무화 한 것은 무엇인가요?"
response = rag_chain.invoke(question)
print(response)

In [52]:
response

NameError: name 'response' is not defined